## QAT INT8 TensorRT Validation (Holdout Set)

Same pipeline as `04_2_qat_trt_test` but on the 5k holdout-val split instead of imagenet2.
Pipeline: load QAT checkpoint → export QDQ ONNX → build TRT engine → run inference.

In [1]:
SKIP_EXISTING = False
OUTPUT_ROOT = "/home/pf4636/code/resnet/quantized_resnets/runs/val/qat"

In [2]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
QAT_MOD = PYFILES / "qat_modelopt"
for p in [str(PYFILES), str(QAT_MOD)]:
    if p not in sys.path:
        sys.path.insert(0, p)

In [3]:
import json
import time
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import onnx
import modelopt.torch.opt as mto

from torch.utils.data import DataLoader

from src.config import ExperimentConfig, set_seed
from src.data import build_train_holdout_split, build_imagenet_transform
from trt.trt_builder import build_engine
from trt.trt_infer import trt_evaluate
from quantize import get_model, restore_modelopt_state

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.3f}".format)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.10.0+cu130 | cuda: True


In [4]:
QAT_CKPT_ROOT  = PROJECT_ROOT / ".checkpoints" / "qat"
FP32_CKPT_ROOT = PROJECT_ROOT / ".checkpoints"
ONNX_DIR       = PROJECT_ROOT / "onnx" / "qat"
ENGINE_DIR     = PROJECT_ROOT / "engines" / "qat"
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
INPUT_BITS     = [8, 4, 2, 1]

checkpoints = {}
skipped = []
for bits in INPUT_BITS:
    run_name = f"int8_in{bits}b"
    run_dir = QAT_CKPT_ROOT / run_name
    if not run_dir.exists():
        skipped.append(f"int8/in{bits}b (no dir)")
        continue
    for seed_dir in sorted(run_dir.iterdir()):
        if not seed_dir.is_dir() or not seed_dir.name.startswith("seed_"):
            continue
        weights = seed_dir / "qat_modelopt_best.pth"
        mostate = seed_dir / "qat_modelopt_best_mostate.pt"
        seed_num = int(seed_dir.name.split("_")[1])
        fp32_ckpt = FP32_CKPT_ROOT / f"fp32_{bits}bit" / seed_dir.name / "best.pth"
        if not weights.exists() or not mostate.exists() or not fp32_ckpt.exists():
            skipped.append(f"int8/in{bits}b/{seed_dir.name}")
            continue
        checkpoints[(bits, seed_dir.name)] = {
            "weights": weights,
            "mostate": mostate,
            "fp32_ckpt": fp32_ckpt,
            "seed": seed_num,
        }

print(f"QAT INT8 checkpoints found: {len(checkpoints)}")
for (bits, seed), paths in checkpoints.items():
    print(f"  in{bits}b / {seed}")
if skipped:
    print(f"\nSkipped (not complete): {', '.join(skipped)}")

QAT INT8 checkpoints found: 12
  in8b / seed_1
  in8b / seed_2
  in8b / seed_42
  in4b / seed_1
  in4b / seed_2
  in4b / seed_42
  in2b / seed_1
  in2b / seed_2
  in2b / seed_42
  in1b / seed_1
  in1b / seed_2
  in1b / seed_42


In [5]:
def load_qat_model(bits: int, seed_name: str) -> nn.Module:
    paths = checkpoints[(bits, seed_name)]
    model = get_model(str(paths["fp32_ckpt"]), num_classes=100)
    restore_modelopt_state(model, str(paths["mostate"]))
    ckpt = torch.load(paths["weights"], map_location="cpu")
    state = ckpt["model"] if "model" in ckpt else ckpt
    model.load_state_dict(state)
    model.eval()
    return model

def export_qat_onnx(model: nn.Module, onnx_path: Path):
    onnx_path.parent.mkdir(parents=True, exist_ok=True)
    model.eval().cpu()
    dummy = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        torch.onnx.export(
            model, (dummy,), str(onnx_path),
            input_names=["images"],
            output_names=["logits"],
            opset_version=17,
            dynamic_axes={"images": {0: "batch"}, "logits": {0: "batch"}},
            export_params=True,
            do_constant_folding=True,
            dynamo=False,
        )
    print(f"    ONNX saved ({onnx_path.stat().st_size / 1e6:.1f} MB)")

def build_val_loader(cfg: ExperimentConfig) -> DataLoader:
    """Holdout-val split (5k samples from training data)."""
    transform = build_imagenet_transform(cfg)
    _, holdout = build_train_holdout_split(
        data_root=cfg.imagenet_path,
        num_classes=cfg.num_classes,
        val_per_class=50,
        seed=42,
        eval_transform=transform,
    )
    return DataLoader(
        holdout,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
        drop_last=True,
    )

## Export QDQ ONNX

The Q/DQ nodes from ModelOpt's fake-quant graph are preserved in the ONNX export, so TRT gets the scales directly.

In [6]:
for (bits, seed_name), paths in checkpoints.items():
    onnx_path = ONNX_DIR / f"int8_in{bits}b" / seed_name / "resnet18_qat_int8_qdq.onnx"

    if SKIP_EXISTING and onnx_path.exists():
        print(f"  int8/in{bits}b/{seed_name}: ONNX exists, skipping")
        continue

    print(f"\n  Exporting int8/in{bits}b/{seed_name} ...")
    model = load_qat_model(bits, seed_name)
    export_qat_onnx(model, onnx_path)
    del model
    torch.cuda.empty_cache()

print("\nAll QAT ONNX exports ready.")


  Exporting int8/in8b/seed_1 ...
Inserted 107 quantizers


/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/modelopt/torch/quantization/nn/modules/quant_module.py:59: UserWarning: Could not identify the device for TensorQuantizer states of maxpool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(
/home/pf4636/code/resnet/quantized_resnets/.venv/lib/python3.12/site-packages/modelopt/torch/quantization/nn/modules/quant_module.py:59: UserWarning: Could not identify the device for TensorQuantizer states of pool. Please move the model to the right device now. This can be done by calling `model.to(device)`.
  warnings.warn(
/tmp/ipykernel_26756/1634319773.py:16: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorc

    ONNX saved (45.0 MB)

  Exporting int8/in8b/seed_2 ...
Inserted 107 quantizers
    ONNX saved (45.0 MB)

  Exporting int8/in8b/seed_42 ...
Inserted 107 quantizers
    ONNX saved (45.0 MB)

  Exporting int8/in4b/seed_1 ...
Inserted 107 quantizers
    ONNX saved (45.0 MB)

  Exporting int8/in4b/seed_2 ...
Inserted 107 quantizers
    ONNX saved (45.0 MB)

  Exporting int8/in4b/seed_42 ...
Inserted 107 quantizers
    ONNX saved (45.0 MB)

  Exporting int8/in2b/seed_1 ...
Inserted 107 quantizers
    ONNX saved (45.0 MB)

  Exporting int8/in2b/seed_2 ...
Inserted 107 quantizers
    ONNX saved (45.0 MB)

  Exporting int8/in2b/seed_42 ...
Inserted 107 quantizers
    ONNX saved (45.0 MB)

  Exporting int8/in1b/seed_1 ...
Inserted 107 quantizers
    ONNX saved (45.0 MB)

  Exporting int8/in1b/seed_2 ...
Inserted 107 quantizers
    ONNX saved (45.0 MB)

  Exporting int8/in1b/seed_42 ...
Inserted 107 quantizers
    ONNX saved (45.0 MB)

All QAT ONNX exports ready.


In [7]:
print("Q/DQ node counts:")
for (bits, seed_name) in checkpoints:
    path = ONNX_DIR / f"int8_in{bits}b" / seed_name / "resnet18_qat_int8_qdq.onnx"
    if not path.exists():
        print(f"  int8/in{bits}b/{seed_name}: NOT FOUND")
        continue
    m = onnx.load(str(path), load_external_data=False)
    ops = [n.op_type for n in m.graph.node]
    print(f"  int8/in{bits}b/{seed_name}: Q={ops.count('QuantizeLinear'):3d}  DQ={ops.count('DequantizeLinear'):3d}  size={path.stat().st_size/1e6:.1f}MB")

Q/DQ node counts:
  int8/in8b/seed_1: Q= 44  DQ= 44  size=45.0MB
  int8/in8b/seed_2: Q= 44  DQ= 44  size=45.0MB
  int8/in8b/seed_42: Q= 44  DQ= 44  size=45.0MB
  int8/in4b/seed_1: Q= 44  DQ= 44  size=45.0MB
  int8/in4b/seed_2: Q= 44  DQ= 44  size=45.0MB
  int8/in4b/seed_42: Q= 44  DQ= 44  size=45.0MB
  int8/in2b/seed_1: Q= 44  DQ= 44  size=45.0MB
  int8/in2b/seed_2: Q= 44  DQ= 44  size=45.0MB
  int8/in2b/seed_42: Q= 44  DQ= 44  size=45.0MB
  int8/in1b/seed_1: Q= 44  DQ= 44  size=45.0MB
  int8/in1b/seed_2: Q= 44  DQ= 44  size=45.0MB
  int8/in1b/seed_42: Q= 44  DQ= 44  size=45.0MB


## Build TRT Engines & Run Inference

In [8]:
OUT_DIR = Path(OUTPUT_ROOT).resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

all_records = []
criterion = nn.CrossEntropyLoss()

for (bits, seed_name), paths in checkpoints.items():
    seed_num = paths["seed"]
    run_id = f"trt_qat_int8_b{bits}_cuda"
    result_path = OUT_DIR / seed_name / run_id / "result.json"

    if SKIP_EXISTING and result_path.exists():
        print(f"  int8/in{bits}b/{seed_name}: exists, skipping")
        with open(result_path) as f:
            all_records.append(json.load(f))
        continue

    print(f"\n--- QAT INT8 TRT, input_bits={bits}, {seed_name} ---")

    onnx_path = ONNX_DIR / f"int8_in{bits}b" / seed_name / "resnet18_qat_int8_qdq.onnx"
    engine_path = ENGINE_DIR / f"int8_in{bits}b" / seed_name / f"{run_id}.engine"

    if not engine_path.exists():
        print(f"  Building TRT engine from {onnx_path.name} ...")
        engine_path.parent.mkdir(parents=True, exist_ok=True)
        build_engine(
            onnx_path=onnx_path,
            engine_path=engine_path,
            precision="int8",
            batch_size=1,
            workspace_mb=2048,
        )
    else:
        print(f"  Engine exists: {engine_path.name}")

    cfg = ExperimentConfig(
        backend="tensorrt",
        device="cuda",
        batch_size=1,
        seed=seed_num,
        num_eval_batches=None,
        input_quant_bits=bits,
        model_precision="int8",
    )
    set_seed(cfg)

    val_loader = build_val_loader(cfg)

    t0 = time.perf_counter()
    tracker = trt_evaluate(engine_path, cfg, val_loader, criterion)
    elapsed = time.perf_counter() - t0

    r = tracker.summary()
    print(f"  top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.3f}ms")

    payload = {
        "status": "ok",
        "run_id": run_id,
        "system": cfg.stamp(),
        "config": cfg.to_dict(),
        "config_extra": {
            "qat_precision": "int8",
            "input_quant_bits": bits,
            "seed": seed_num,
            "backend": "tensorrt",
            "split": "holdout_val",
        },
        "results": r,
        "error": None,
        "total_eval_time_sec": round(elapsed, 3),
    }

    result_path.parent.mkdir(parents=True, exist_ok=True)
    with open(result_path, "w") as f:
        json.dump(payload, f, indent=2, sort_keys=True)

    all_records.append(payload)
    torch.cuda.empty_cache()

print(f"\n{len(all_records)} runs complete.")


--- QAT INT8 TRT, input_bits=8, seed_1 ---
  Building TRT engine from resnet18_qat_int8_qdq.onnx ...
[trt_builder] Building engine | precision=int8 | batch=1 | workspace=2048 MiB
[trt_builder] Engine saved: /home/pf4636/code/resnet/quantized_resnets/engines/qat/int8_in8b/seed_1/trt_qat_int8_b8_cuda.engine (12.1 MB)
[data] Train: 124,395  Holdout-Val: 5,000  (num_classes=100, val_per_class=50, seed=42)
[trt_infer] Engine: /home/pf4636/code/resnet/quantized_resnets/engines/qat/int8_in8b/seed_1/trt_qat_int8_b8_cuda.engine
[trt_infer] Input: 'images'  Output: 'logits'  Dynamic batch: False
[trt_infer] Evaluating 5000 batches (first 30 are warmup) ...
[trt_infer] --- Warmup complete (30 batches) — starting metric collection ---
  [40/5000]  Top-1: 100.00%  Top-5: 100.00%  Infer: 1.01 ms/batch
  [50/5000]  Top-1: 100.00%  Top-5: 100.00%  Infer: 0.96 ms/batch
  [60/5000]  Top-1: 100.00%  Top-5: 100.00%  Infer: 0.91 ms/batch
  [70/5000]  Top-1: 100.00%  Top-5: 100.00%  Infer: 0.99 ms/batch
  

## Per-Run Results

In [9]:
rows = []
for rec in all_records:
    r = rec["results"]
    extra = rec.get("config_extra", rec.get("config", {}))
    bits = extra.get("input_quant_bits", rec["config"].get("input_quant_bits", None))
    seed = extra.get("seed", 42)
    rows.append({
        "method": "qat_int8_trt",
        "input_bits": bits,
        "seed": seed,
        "top1": r["top1_acc"],
        "top5": r["top5_acc"],
        "lat_ms": r["infer_ms_avg"],
        "lat_std": r.get("infer_ms_std", None),
        "throughput": r.get("throughput_infer_sps", None),
    })

df_all = pd.DataFrame(rows).sort_values(
    ["input_bits", "seed"], ascending=[True, True]
).reset_index(drop=True)
df_all

,method,input_bits,seed,top1,top5,lat_ms,lat_std,throughput
0,qat_int8_trt,1,1,87.968,97.686,0.730,0.317,1370.057
1,qat_int8_trt,1,2,88.068,98.089,0.739,0.312,1353.782
2,qat_int8_trt,1,42,75.191,92.837,0.718,0.309,1392.493
3,qat_int8_trt,2,1,93.199,99.316,0.656,0.265,1525.174
4,qat_int8_trt,2,2,92.797,99.517,0.655,0.244,1526.548
5,qat_int8_trt,2,42,81.388,95.453,0.685,0.278,1459.478
6,qat_int8_trt,4,1,96.942,99.819,0.739,0.296,1353.238
7,qat_int8_trt,4,2,96.640,99.799,0.724,0.299,1381.509
8,qat_int8_trt,4,42,82.535,95.513,0.666,0.286,1502.450
9,qat_int8_trt,8,1,97.123,99.799,0.754,0.320,1326.460


## Averaged Results (mean ± std across seeds)

In [10]:
avg_df = df_all.groupby(["method", "input_bits"]).agg(
    top1_mean=("top1", "mean"),
    top1_std=("top1", "std"),
    top5_mean=("top5", "mean"),
    top5_std=("top5", "std"),
    infer_ms_mean=("lat_ms", "mean"),
    infer_ms_std=("lat_ms", "std"),
    throughput_mean=("throughput", "mean"),
    n_seeds=("seed", "count"),
).round(3).reset_index()

avg_df = avg_df.sort_values("input_bits").reset_index(drop=True)
avg_df

,method,input_bits,top1_mean,top1_std,top5_mean,top5_std,infer_ms_mean,infer_ms_std,throughput_mean,n_seeds
0,qat_int8_trt,1,83.742,7.406,96.204,2.923,0.729,0.010,1372.111,3
1,qat_int8_trt,2,89.128,6.706,98.095,2.291,0.665,0.017,1503.733,3
2,qat_int8_trt,4,92.039,8.232,98.377,2.480,0.709,0.039,1412.399,3
3,qat_int8_trt,8,91.992,8.800,98.404,2.451,0.732,0.033,1367.333,3
